In [1]:
import math

# normal CDF and PDF 
def N(x):  # standard normal CDF
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def phi(x):  # standard normal PDF
    return (1.0 / math.sqrt(2.0 * math.pi)) * math.exp(-0.5 * x * x)

# Black–Scholes call delta with continuous dividend yield 
def call_delta(S, K, r, q, sigma, T):
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    return math.exp(-q * T) * N(d1), d1  # return delta and d1 (for reuse)

# Newton's method to find K so that delta = target_delta 
def strike_for_delta(S, r, q, sigma, T, target_delta=0.5, K0=None, tol=1e-6, max_iter=50):
    if K0 is None:
        K0 = S  # ATM start by default

    K = float(K0)
    print(f"iter 0: K = {K:.12f}")

    for it in range(1, max_iter + 1):
        delta, d1 = call_delta(S, K, r, q, sigma, T)

        f = delta - target_delta

        # f'(K) = d/dK [e^{-qT} N(d1(K))] = e^{-qT} * phi(d1) * d(d1)/dK
        # d(d1)/dK = -1 / (K * sigma * sqrt(T))
        fp = -math.exp(-q * T) * phi(d1) / (K * sigma * math.sqrt(T))

        K_new = K - f / fp
        print(f"iter {it}: K = {K_new:.12f}   |ΔK| = {abs(K_new - K):.3e}")

        if abs(K_new - K) < tol:
            return K_new

        K = K_new

    return K  # return last iterate if max_iter hit

# data
S = 30.0
sigma = 0.30
q = 0.01
r = 0.025
T = 3/12  
K_star = strike_for_delta(S, r, q, sigma, T, target_delta=0.5, K0=S, tol=1e-6)
print("\nFinal strike K* =", f"{K_star:.6f}")

iter 0: K = 30.000000000000
iter 1: K = 30.437314816940   |ΔK| = 4.373e-01
iter 2: K = 30.439064456186   |ΔK| = 1.750e-03
iter 3: K = 30.439064505337   |ΔK| = 4.915e-08

Final strike K* = 30.439065
